# Three worlds where the obvious regression is wrong

Each of the three simulated worlds below has a known true effect, twenty thousand rows, and
no missing data. In each of them, the regression a competent analyst writes first returns a
confident, precise, wrong answer. What changes between them is *why* — a confounder, a
feedback loop into the treatment, a latent that no covariate can reach — and therefore which
estimator is licensed.

The graph names the route; these are the estimators for each. All are pure numpy with
classical standard errors, return a `LinearEstimate` spec, and their intervals are labelled
`wald` — a frequentist CI is a different object from a credible interval and the type says so.

In [ ]:
from axiom.identify import (
    CausalGraph, EndogeneityTest, FrontDoorRoute, InstrumentRoute, LinearEstimate,
    conditional_instruments, durbin_wu_hausman, frontdoor_admissible, frontdoor_linear,
    frontdoor_sets, hausman_iv_vs_ols, identify, instrument_admissible, instruments, ols,
    two_stage_least_squares, weak_instrument_check,
)
from axiom.sim import confounded_world, frontdoor_world, iv_world

from axiom.display import enable, show, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, intervals

enable();  # every axiom result renders itself from here on

## Front-door criterion

In [ ]:
from axiom.display import show

fd_graph = CausalGraph.from_edges("X -> M, M -> Y, X <-> Y")
print(frontdoor_admissible(fd_graph, "X", "Y", ["M"]), frontdoor_sets(fd_graph, "X", "Y"))
route = FrontDoorRoute(mediators=("M",), treatment="X", outcome="Y")
show(route)
# a latent between X and M breaks condition (ii)
print(frontdoor_admissible(CausalGraph.from_edges("X -> M, M -> Y, X <-> Y, X <-> M"), "X", "Y", ["M"]))

## Instruments

`instruments` lists unconditional instruments (relevance in $G$, exclusion in $G_{\underline{X}}$);
`conditional_instruments` lists $(Z, W)$ pairs where the exclusion holds only given $W$.

In [ ]:
iv_graph = CausalGraph.from_edges("Z -> X, X -> Y, X <-> Y")
print(instruments(iv_graph, "X", "Y"), instrument_admissible(iv_graph, "X", "Y", "Z"))
print(instruments(CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y, X <-> Y"), "X", "Y"), "<- exclusion violated")

civ = CausalGraph.from_edges("W -> Z, W -> Y, Z -> X, X -> Y, X <-> Y")
print(instruments(civ, "X", "Y"), conditional_instruments(civ, "X", "Y"))
print(InstrumentRoute(instrument="Z", conditioning=("W",), treatment="X", outcome="Y"))

## Estimating along the licensed route

Each `sim` world carries its truth. The naive estimate is biased by construction; the
estimator matching the verdict's route recovers the truth within its standard error.

In [ ]:
results = []

world = confounded_world()
frame = world.observed(world.simulate(20_000, seed=0))
truth = world.total_effect("X", "Y")
v = identify(world.graph, "X", "Y")

naive: LinearEstimate = ols(frame, "Y", "X")
adjusted = ols(frame, "Y", "X", covariates=v.adjustment_set)
print(f"truth {truth} | naive {naive.estimate:.3f} ± {naive.se:.3f} | adjusted {adjusted.estimate:.3f} ± {adjusted.se:.3f}")
print(adjusted.ci(0.95), adjusted.method, adjusted.covariates)
results += [("confounded · ols", naive, truth), (f"confounded · adjusted for {v.adjustment_set}", adjusted, truth)]

In [ ]:
world = iv_world()
frame = world.observed(world.simulate(20_000, seed=1))
v = identify(world.graph, "X", "Y")
iv = two_stage_least_squares(frame, "Y", "X", instruments=[v.instrument])
print(f"truth {world.total_effect('X', 'Y')} | ols {ols(frame, 'Y', 'X').estimate:.3f} | 2sls {iv.estimate:.3f} ± {iv.se:.3f}")
print(iv.detail)
strength = weak_instrument_check(iv)
print(strength.name, strength.state, strength.statement)
results += [("instrumented · ols", ols(frame, "Y", "X"), world.total_effect("X", "Y")),
            ("instrumented · 2sls", iv, world.total_effect("X", "Y"))]

In [ ]:
world = frontdoor_world()
frame = world.observed(world.simulate(20_000, seed=2))
v = identify(world.graph, "X", "Y")
fd = frontdoor_linear(frame, "Y", "X", mediators=v.mediators)
print(f"truth {world.total_effect('X', 'Y')} | ols {ols(frame, 'Y', 'X').estimate:.3f} | front-door {fd.estimate:.3f} ± {fd.se:.3f}")
print(fd.ci(0.9))
results += [("front-door world · ols", ols(frame, "Y", "X"), world.total_effect("X", "Y")),
            ("front-door world · front-door", fd, world.total_effect("X", "Y"))]

In [ ]:
fig = intervals(
    [(label, est.estimate - tr, est.estimate - tr - 2 * est.se, est.estimate - tr + 2 * est.se)
     for label, est, tr in results],
    ref=0.0, ref_label="the truth",
    title="Six estimates, three worlds, one number each is trying to hit",
    subtitle="estimate minus the known truth, ±2 classical standard errors",
    x_title="error",
)
caption(fig, "Every second row sits on the line; every first row misses it by many standard "
             "errors. The estimator did not change between the pairs — the question of which "
             "one the graph licenses did.")

## Is the IV route needed? Endogeneity tests

An instrument is not free: 2SLS uses only the variation in the treatment that the instrument
explains, so its standard error is larger — twice OLS's on this world. Running it when
ordinary least squares was already unbiased is a real cost, and these tests are how you find
out whether you are paying it for nothing.

`durbin_wu_hausman` is the control-function form; `hausman_iv_vs_ols` contrasts the two
estimates directly. A degenerate contrast (negative variance difference) returns a typed
`Unverified`, not a fabricated p-value.

In [ ]:
world = iv_world()
frame = world.observed(world.simulate(20_000, seed=1))
dwh = durbin_wu_hausman(frame, "Y", "X", instruments=["Z"])
print(dwh if not isinstance(dwh, EndogeneityTest) else (dwh.conclusion, round(dwh.statistic, 2), dwh.p_value))

h = hausman_iv_vs_ols(ols(frame, "Y", "X"), two_stage_least_squares(frame, "Y", "X", instruments=["Z"]))
print(h if not isinstance(h, EndogeneityTest) else (h.conclusion, round(h.statistic, 2)))

## Guard rails

The estimators refuse the outcome on the right-hand side, missing columns, and too few rows —
loudly, never with a confidently wrong number. Regressing an outcome on itself is a typo that
produces an R² of 1.0 and a slide that nobody questions.

In [ ]:
refused = []
for label, bad in (
    ("outcome used as its own covariate", lambda: ols(frame, "Y", "X", covariates=["Y"])),
    ("a column that is not there", lambda: ols(frame, "Y", "nope")),
    ("fewer rows than parameters", lambda: ols(frame.head(2), "Y", "X")),
):
    try:
        bad()
    except (ValueError, KeyError) as e:
        refused.append([label, type(e).__name__, str(e)[:90]])
table(refused, headers=("call", "raised", "why"))

## Assigned is not received

Every estimator above assumes the treatment column is the treatment. In a field experiment it
usually is not: some units assigned to treatment never get it, and some units assigned to
control get it anyway. `diagnose.delivery` measures that gap. What it cannot do is tell you
what to estimate, because there is no longer one thing to estimate — there are two.

    intention to treat   the effect of being *assigned*, over everybody
    complier effect      the effect of being *exposed*, over the units the
                         assignment actually moved

Both are causal and both are identified by the randomization. They are not two estimators of
one number: they differ on two facets of the charter's table at once — `intervention`
(assigned against received) and `population` (everyone against the compliers). A house that
pools one party's intention-to-treat with another party's complier effect has made exactly the
silent facet difference the estimand machinery exists to prevent.

In [ ]:
import numpy as np
import pandas as pd

from axiom.identify import (
    SELECTION_MONOTONICITY, LeeBounds, lee_bounds,
    EXCLUSION, MONOTONICITY, ComplianceReport, ComplianceTable,
    compliance, compliance_table, complier_effect, intention_to_treat,
)

EFFECT, SHARE, ALWAYS = 2.0, 0.6, 0.05
rng_c = np.random.default_rng(0)
n_c = 4000
u = rng_c.random(n_c)
complier, always = u < SHARE, (u >= SHARE) & (u < SHARE + ALWAYS)
assigned = (rng_c.random(n_c) < 0.5).astype(float)
exposed = np.where(always, 1.0, np.where(complier, assigned, 0.0))
# the outcome depends on exposure and never on assignment: the exclusion restriction holds
field = pd.DataFrame({"y": 10.0 + EFFECT * exposed + rng_c.normal(0, 1, n_c),
                      "assigned": assigned, "exposed": exposed})

cross: ComplianceTable = compliance_table(field, "assigned", "exposed")
print(cross.ledger_line().statement)
table_rows = [["compliers", f"{cross.complier_share:.3f}", "exposed if assigned, not if not"],
              ["always-takers", f"{cross.always_taker_share:.3f}", "exposed either way"],
              ["never-takers", f"{cross.never_taker_share:.3f}", "exposed neither way"]]
table(table_rows, headers=("type", "share", "what it means"),
              title="the population, sized without a single member being named")

### Two numbers, and the arithmetic between them

The intention-to-treat effect is diluted by everyone the assignment did not move; the complier
effect is what happened to the ones it did. They stand in the relation
`itt = complier × share`, which is exactly why reading one as the other is so easy and so
wrong: at 61 % compliance an intention-to-treat effect of 1.23 and a complier effect of 2.01
are the same experiment.

In [ ]:
report: ComplianceReport = compliance(field, "y", "assigned", "exposed")
assert report.itt == intention_to_treat(field, "y", "assigned")
assert report.complier == complier_effect(field, "y", "exposed", "assigned")
print(report.summary())
print(f"\nitt / share = {report.itt.estimate / report.share:.4f}"
      f"  |  complier effect = {report.complier.estimate:.4f}  |  truth = {EFFECT}")

verdict = report.verdict()
print("\nverdict:", verdict.status, "via", verdict.route)
print(" ", verdict.reason)
table([[a.name, a.state, a.challenged_by] for a in report.assumptions()],
              headers=("assumption", "state", "challenged by"),
              title="what the complier effect needs beyond the randomization")

### `downgraded`, never `identified`

The exclusion restriction cannot be checked in the data and never will be, so the strongest
honest verdict for a complier effect is `downgraded` under named assumptions. Monotonicity is
untestable too, but it has one implication the data *can* check — the complier share cannot be
negative — and when that fails the verdict is `blocked` rather than downgraded.

The one case that is `identified` is perfect compliance, where being assigned and being
exposed are the same event and there is no second quantity at all.

In [ ]:
cases = {
    "as run": field,
    "perfect compliance": field.assign(exposed=field["assigned"]),
    "assignment moved nothing": field.assign(exposed=0.0),
    "defiers (control exposed more)": field.assign(exposed=1.0 - field["exposed"]),
}
rows = []
for label, frame in cases.items():
    r = compliance(frame, "y", "assigned", "exposed")
    v = r.verdict()
    rows.append([label, f"{r.share:+.3f}", f"{r.itt.estimate:+.3f}",
                 "—" if r.complier is None else f"{r.complier.estimate:+.3f}",
                 v.status, v.route])
table(rows, headers=("case", "complier share", "itt", "complier", "verdict", "route"))
print("\nperfect compliance:", compliance(cases["perfect compliance"], "y", "assigned",
                                          "exposed").verdict().reason)

### A weak first stage does not hide

The complier effect is 2SLS of the outcome on exposure instrumented by assignment — the same
`two_stage_least_squares` used above, not a second implementation of the Wald ratio — so it
carries a first-stage F and `weak_instrument_check` comes with it. When the assignment barely
moved exposure, the ratio's denominator is nearly zero and the interval says so.

In [ ]:
thin_u = rng_c.random(600)
thin_assigned = (rng_c.random(600) < 0.5).astype(float)
thin_exposed = np.where(thin_u < 0.02, thin_assigned, 0.0)
thin = pd.DataFrame({"y": 10.0 + EFFECT * thin_exposed + rng_c.normal(0, 1, 600),
                     "assigned": thin_assigned, "exposed": thin_exposed})
weak = compliance(thin, "y", "assigned", "exposed")
print(f"complier share {weak.share:.4f} against {report.share:.4f} in the field experiment")
table(
    [["field experiment", f"{report.share:.3f}", f"{report.complier.se:.3f}",
      report.complier.ci(0.95).text(), report.instrument_strength.state],
     ["thin first stage", f"{weak.share:.3f}", f"{weak.complier.se:.3f}",
      weak.complier.ci(0.95).text(), weak.instrument_strength.state]],
    headers=("case", "share", "se", "95% interval", "instrument_strength"))
print("\n" + report.ledger()[1].statement)

### When the outcome is missing, not the treatment

Non-compliance leaves you two estimands and both are estimable. **Attrition** leaves you
neither, because there is no number for the units that dropped out and they are not a random
subset of the ones that stayed.

What survives is a bound. Under monotonicity in selection — assignment can only push reporting
one way — the reporting units of the arm with the *higher* response rate are a known mixture:
always-responders, who would have reported either way, and marginal ones, who reported only
because of their assignment. Trimming the top of that arm's outcome distribution by the
marginal share gives the lowest the always-responder effect can be; trimming the bottom gives
the highest (Lee 2009).

In [ ]:
rng_a = np.random.default_rng(0)
n_a = 6000
assigned_a = (rng_a.random(n_a) < 0.5).astype(float)
u = rng_a.normal(size=n_a)                       # latent quality
outcome = 10.0 + 1.0 * assigned_a + u            # the always-responder effect is exactly 1.0
always = u > -0.5                                # would report either way
marginal = (u > -1.2) & (u <= -0.5)              # report only if assigned to treatment
reported = np.where(always, 1.0, np.where(marginal, assigned_a, 0.0))
attrited = pd.DataFrame({"y": np.where(reported == 1.0, outcome, -999.0),   # never read
                         "assigned": assigned_a, "reported": reported})

bounds: LeeBounds = lee_bounds(attrited, "y", "assigned", "reported")
print(bounds.summary())
print("\ntruth: 1.0 — inside the bounds:", bounds.lower <= 1.0 <= bounds.upper)
print("the naive contrast over reporting units:", round(bounds.naive, 4), "— outside by",
      round(abs(bounds.naive - 1.0), 4))

The naive contrast is wrong by almost thirty per cent of the effect, and it is wrong in a knowable
direction: the units treatment pulled into reporting are low-outcome ones, so they drag the
treated mean down. The bounds hold whatever those missing outcomes were.

Two things the result says that a bare pair of numbers would not. Its **population** is a
latent one — always-responders are sized by the two response rates and can never be listed —
so an estimand built on it inherits the refusal `meta.pool` learned in note 0032. And its
verdict is `downgraded` even where the bounds collapse, because selection monotonicity is not
testable: the units that would falsify it are exactly the ones never observed in one arm.

In [ ]:
print("population:", bounds.population.latent)
print("latent:", bounds.population.is_latent, "| trimmed", f"{bounds.trimmed:.1%}",
      "of the", bounds.trimmed_arm, "arm")
print("\nverdict:", bounds.verdict().status, "under",
      [a.name for a in bounds.verdict().assumptions])
print(" ", SELECTION_MONOTONICITY.challenged_by)
show(bounds.ledger_line())

## What this bought you

Four estimators that each know which verdict licenses them, checked against worlds whose
truth is known — and a picture of what the licensed route buys, in the units of the effect
itself, rather than an argument about methodology.